In [ ]:
import sys, os, glob, shutil, time, json, gc
import numpy as np, pandas as pd
t0 = time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:6.0f}с] {m}", flush=True)
code = os.path.dirname(glob.glob("/kaggle/input/**/pair_features.py", recursive=True)[0])
os.makedirs("/kaggle/working/src", exist_ok=True)
for p in glob.glob(code + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
open("/kaggle/working/src/__init__.py", "a").close()
os.makedirs("/kaggle/working/models", exist_ok=True)
for p in glob.glob(code + "/*.json"): shutil.copy(p, "/kaggle/working/models/")
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")
from src.pair_features import build_matrix, feature_names
from src.measure_features import measures, compare_measures, MEASURE_FEATURES
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
from scipy.stats import rankdata

fold = os.path.dirname(glob.glob("/kaggle/input/**/llm_valid_pairs.parquet", recursive=True)[0])
sc = os.path.dirname(glob.glob("/kaggle/input/**/ce_relaxed.npy", recursive=True)[0])
pairs = pd.read_parquet(fold + "/llm_valid_pairs.parquet")
items = pd.read_parquet(fold + "/llm_valid_items.parquet")
y = (pairs["target"].to_numpy() > 0).astype(np.int8)
cat = pairs["id1"].map(dict(zip(items.id, items.category.astype(str)))).astype(str).to_numpy()
log(f"пар {len(pairs):,}, карточек {len(items):,}, доля+ {y.mean():.3f}")

names = list(feature_names(False))
X = np.zeros((len(pairs), len(names)), dtype=np.float32)
pc = cat
for c in sorted(set(items.category.astype(str))):
    rows = np.flatnonzero(pc == c)
    if not len(rows): continue
    sub = items[items.category.astype(str) == c].reset_index(drop=True)
    t = time.perf_counter()
    X[rows] = build_matrix(sub, pairs.iloc[rows].reset_index(drop=True), with_neighbours=False)
    log(f"  {c}: {len(rows):,} пар за {time.perf_counter()-t:.0f}с")
    del sub; gc.collect()
log("признаки готовы")

M = {int(i): measures(a) for i, a in zip(items.id, items.attributes)}
M_X = np.array([[r[n] for n in MEASURE_FEATURES] for r in
                (compare_measures(M[x], M[z]) for x, z in zip(pairs.id1, pairs.id2))], dtype=np.float32)
CE = {n: np.load(f"{sc}/{n}.npy") for n in ("ce_relaxed", "ce_combo", "ce_spec", "ce_self")}
FM = np.load(sc + "/feature_model.npy")
masks = {c: cat == c for c in np.unique(cat)}
def rk(s):
    o = np.empty(len(s), np.float32)
    for m in masks.values(): o[m] = rankdata(s[m]) / m.sum()
    return o
enc = 0.56*rk(CE["ce_relaxed"]) + 0.44*rk(CE["ce_combo"])
SHARE = json.loads(r"""{"Автотовары": 0.35, "Аптека": 0.15, "Бытовая техника": 0.3, "Бытовая химия": 0.15, "Галантерея и аксессуары": 0.4, "Детские товары": 0.3, "Дом и сад": 0.25, "Канцелярские товары": 0.35, "Красота и гигиена": 0.35, "Мебель": 0.2, "Музыкальные инструменты": 0.2, "Обувь": 0.0, "Одежда": 0.0, "Продукты питания": 0.1, "Спорт и отдых": 0.3, "Строительство и ремонт": 0.2, "Товары для животных": 0.3, "Хобби и творчество": 0.35, "Электроника": 0.2, "Ювелирные изделия": 0.15}""")
blend = enc.copy()
for c, m in masks.items():
    k = float(SHARE.get(c, 0.0)); blend[m] = (1-k)*enc[m] + k*rk(FM)[m]

def macro(s, rate=0.111, seeds=8):
    vals = []
    for seed in range(seeds):
        rng = np.random.default_rng(seed); per = []
        for m in masks.values():
            rows = np.flatnonzero(m); pos, neg = rows[y[rows]==1], rows[y[rows]==0]
            keep = min(len(pos), max(5, int(round(rate/(1-rate)*len(neg)))))
            ch = np.concatenate([rng.choice(pos, keep, replace=False), neg])
            per.append(average_precision_score(y[ch], s[ch]))
        vals.append(np.mean(per))
    return float(np.mean(vals)), float(np.std(vals))

half = np.random.default_rng(5).permutation(len(y)) % 2
code_cat = pd.factorize(cat)[0].astype(np.float32)
def oof(mat):
    p = np.zeros(len(y))
    for h in (0, 1):
        tr, te = half != h, half == h
        g = HistGradientBoostingClassifier(max_iter=400, max_leaf_nodes=63,
                                           learning_rate=0.06, random_state=0).fit(mat[tr], y[tr])
        p[te] = g.predict_proba(mat[te])[:, 1]
    return rk(p)

VARIANTS = {
    "A. только признаки (как сейчас)": np.column_stack([X, code_cat]),
    "B. признаки + оценка энкодера":   np.column_stack([X, code_cat, CE["ce_relaxed"], CE["ce_combo"]]),
    "C. B + величины":                 np.column_stack([X, code_cat, CE["ce_relaxed"], CE["ce_combo"], M_X]),
}
mu, sd = macro(blend)
log(f"\nнынешняя смесь рангов: {mu:.6f} ± {sd:.6f}")
log(f"только энкодеры:       {macro(enc)[0]:.6f}")
for tag, mat in VARIANTS.items():
    t = time.perf_counter(); p = oof(mat); m2, s2 = macro(p)
    log(f"{tag:<34} {m2:.6f} ± {s2:.6f}   (обучение {time.perf_counter()-t:.0f}с)")
    # и то же, но смешанное с энкодером рангами — вдруг слияние лучше как участник
    best = max(np.arange(0, 1.01, 0.05), key=lambda w: macro((1-w)*enc + w*p, seeds=3)[0])
    mixed = macro((1-best)*enc + best*p)[0]
    log(f"{'':<34} рангами с энкодером, вес {best:.2f}: {mixed:.6f}")
np.save("/kaggle/working/fold_features.npy", X)
log("готово")
